# 06.8 The Walrus Operator `:=`

Python 3.8 added **assignment expressions**. The `:=` operator assigns a value
*and* evaluates to it, so assignment can finally appear where only expressions
were allowed.

Named the walrus operator because `:=` looks like a walrus lying on its side.

**30 numbered examples.**

## Theory

### Why it was needed

From 03.1: assignment is a **statement**, so it cannot go inside an expression.
That caused a recurring annoyance:

```python
# Either call twice...
if expensive() > 10:
    process(expensive())

# ...or use an extra line
result = expensive()
if result > 10:
    process(result)
```

The walrus fixes it:

```python
if (result := expensive()) > 10:
    process(result)
```

### The rules

1. Assigns and evaluates to the assigned value
2. **Always** requires brackets in most contexts — Python enforces this
3. Cannot assign to attributes or subscripts — `obj.x := 1` is a `SyntaxError`
4. Binds **loosest** of all operators

### Where it genuinely helps

- `while` loops reading until a sentinel
- Avoiding a duplicate call in a condition
- Reusing a computed value inside a comprehension
- Regex match-and-use in one step

### Where it hurts

Anywhere the reader has to hunt for where a name was bound. If a plain
assignment on the previous line is just as clear, use that instead.

In [ ]:
# EXAMPLE 1-6: the basics.
print("EXAMPLE 1-6: what := does")
print("")

# 1. It assigns AND evaluates.
result = (count := 10)
print("   1. result = (count := 10)")
print("      result:", result, " count:", count)

# 2. Inside a condition.
values = [1, 2, 3, 4, 5]
if (total := sum(values)) > 10:
    print("")
    print("   2. if (total := sum(values)) > 10:")
    print("      total is usable in the body:", total)

# 3. The name survives after the statement.
print("")
print("   3. total is still bound afterwards:", total)

# 4. Brackets are usually required.
print("")
print("   4. brackets are enforced:")
try:
    compile("x := 5", "<demo>", "exec")
    print("      bare x := 5 compiled")
except SyntaxError:
    print("      bare `x := 5` is a SyntaxError - use (x := 5)")

# 5. It binds loosest of all operators.
print("")
print("   5. (n := 5) + 1 ->", (n := 5) + 1, " n is", n)
print("      the assignment happened first, then the addition")

# 6. Cannot assign to attributes or items.
print("")
print("   6. what := cannot do:")
for source in ["obj.attr := 1", "items[0] := 1", "a, b := 1, 2"]:
    try:
        compile(source, "<demo>", "exec")
        print(f"      {source:<18} compiled")
    except SyntaxError:
        print(f"      {source:<18} SyntaxError")

In [ ]:
# EXAMPLE 7-12: the while-loop pattern - the best use case.
print("EXAMPLE 7-12: reading until a sentinel")
print("")

# Simulate a stream of chunks ending with an empty one.
def make_reader(chunks):
    """Return a function yielding one chunk per call, then ''."""
    remaining = list(chunks)

    def read():
        return remaining.pop(0) if remaining else ""

    return read


# 7. WITHOUT the walrus - the call is duplicated.
read = make_reader(["alpha", "beta", "gamma"])
collected = []

chunk = read()
while chunk != "":
    collected.append(chunk)
    chunk = read()

print("   7. without walrus - read() appears twice:")
print("      collected:", collected)

# 8. WITH the walrus - one call, one place.
read = make_reader(["alpha", "beta", "gamma"])
collected = []

while (chunk := read()) != "":
    collected.append(chunk)

print("")
print("   8. with walrus - read() appears once:")
print("      collected:", collected)

# 9-12. The same shape appears everywhere.
print("")
print("   9.  reading a file:      while (line := f.readline()):")
print("   10. reading a socket:    while (data := sock.recv(1024)):")
print("   11. polling a queue:     while (job := queue.get()):")
print("   12. consuming an iterator: while (item := next(it, None)):")
print("")
print("   This pattern is why the walrus was added.")

In [ ]:
# EXAMPLE 13-18: avoiding duplicate work.
print("EXAMPLE 13-18: not computing twice")
print("")

call_count = 0


def expensive(value):
    """An expensive computation, counted."""
    global call_count
    call_count += 1
    return value * 2


# 13. Without the walrus - two calls.
call_count = 0
numbers = [1, 5, 10]
results = [expensive(number) for number in numbers if expensive(number) > 5]
print("   13. without walrus:")
print("       results:", results, " calls:", call_count)

# 14. With the walrus - one call per item.
call_count = 0
results = [doubled for number in numbers if (doubled := expensive(number)) > 5]
print("")
print("   14. with walrus:")
print("       results:", results, " calls:", call_count)

# 15. Halved the work.
print("")
print("   15. same answer, half the calls")

# 16-18. Other places it removes duplication.
print("")
print("   16. in a condition:")
data = {"items": [1, 2, 3]}
if (items := data.get("items")) is not None:
    print("       found", len(items), "items")

print("")
print("   17. in a return:")

def first_long_word(words, minimum=5):
    """Return the first word longer than minimum, or None."""
    for word in words:
        if (length := len(word)) > minimum:
            return f"{word} ({length} chars)"
    return None


print("       ", first_long_word(["hi", "hello", "wonderful"]))

print("")
print("   18. in a while with a computed condition:")
budget = 100
spent = 0
purchases = []
costs = [30, 40, 50]
for cost in costs:
    if (remaining := budget - spent - cost) >= 0:
        purchases.append(cost)
        spent += cost
print("       bought", purchases, "with", budget - spent, "left")

In [ ]:
import re

# EXAMPLE 19-24: the regex pattern.
print("EXAMPLE 19-24: match and use in one step")
print("")

lines = [
    "2026-09-13 ERROR disk full",
    "not a log line",
    "2026-09-14 WARN low memory",
]

pattern = re.compile(r"(\d{4}-\d{2}-\d{2}) (\w+) (.+)")

# 19. Without the walrus.
print("   19. without walrus:")
for line in lines:
    match = pattern.match(line)
    if match:
        print(f"       {match.group(2):<6} on {match.group(1)}")

# 20. With the walrus.
print("")
print("   20. with walrus:")
for line in lines:
    if (match := pattern.match(line)):
        print(f"       {match.group(2):<6} on {match.group(1)}")

# 21-24. Other single-expression uses.
print("")
print("   21. in any():")
words = ["short", "tiny", "enormous"]
if any((length := len(word)) > 7 for word in words):
    print("       found a long word, last length checked:", length)

print("")
print("   22. reusing a slice:")
text = "hello world"
if (first_word := text.split()[0]) and len(first_word) > 3:
    print("       first word:", first_word)

print("")
print("   23. in a dict comprehension:")
raw = ["1", "2", "abc", "4"]
parsed = {item: number for item in raw
          if (number := int(item) if item.isdigit() else None) is not None}
print("       ", parsed)

print("")
print("   24. capping a value while keeping the original:")
for candidate in [5, 150]:
    capped = min(value := candidate, 100)
    print(f"       input {value:>3} -> capped {capped}")

In [ ]:
# EXAMPLE 25-30: when NOT to use it.
print("EXAMPLE 25-30: readability limits")
print("")

# 25. Do not use it when a plain line is clearer.
values = [1, 2, 3]
print("   25. POOR:  if (n := len(values)) > 2: ...")
print("       BETTER: n = len(values)")
print("               if n > 2: ...")
print("       The walrus earns its place when it removes DUPLICATION,")
print("       not merely a line.")

# 26. Never nest them.
print("")
print("   26. NEVER: if (a := (b := compute())) > (c := other()):")
print("       Unreadable. Split it up.")

# 27. It does not work at statement level.
print("")
print("   27. as a plain statement, use = not :=")
print("       x = 5      correct")
print("       (x := 5)   legal but pointless")

# 28. Scope leaks out of comprehensions.
print("")
print("   28. the walrus LEAKS out of a comprehension:")
squares = [(square := value ** 2) for value in range(4)]
print("       squares:", squares)
print("       square still bound:", square, "<- the last value")
print("       the loop variable does NOT leak:", end=" ")
try:
    print(value)
except NameError:
    print("NameError, as expected")

# 29. That leak is deliberate and occasionally useful.
print("")
print("   29. deliberate: it lets you keep the last computed value.")

# 30. The guideline.
print("")
print("   30. GUIDELINE: use := when it removes a duplicate call or a")
print("       redundant loop. Otherwise prefer a plain assignment.")

## Takeaways

1. `:=` **assigns and evaluates**, letting assignment appear where only
   expressions are allowed.
2. It exists because assignment is a **statement** (03.1), which blocked a
   recurring pattern.
3. Brackets are **required** in most contexts, and it binds **loosest** of all
   operators.
4. It **cannot** assign to attributes, subscripts, or multiple targets.
5. The best use is the **`while (chunk := read())`** pattern — reading until a
   sentinel without duplicating the call.
6. In comprehensions it avoids computing the same value twice.
7. The assigned name **leaks out** of a comprehension, unlike the loop variable.
8. Use it to remove **duplication**, not merely a line. If a plain assignment
   reads as well, use that.

## Try it yourself

1. Rewrite a `while` loop that currently calls a function twice.
2. Write a comprehension that filters and transforms with one call per item.
3. Try `obj.attr := 1`. Why is it a `SyntaxError`?
4. Confirm the walrus name leaks out of a comprehension but the loop variable
   does not.
5. Find code of your own where `:=` genuinely helps — and one where it would not.